## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.


In [1]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent
from agents import OpenAIChatCompletionsModel, Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

In [5]:
gemini_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openrouter_base_url = "https://openrouter.ai/api/v1"

gemini_api_key = os.getenv("GOOGLE_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

gemini_client = AsyncOpenAI(api_key= gemini_api_key, base_url= gemini_base_url)
openrouter_client = AsyncOpenAI(api_key= openrouter_api_key, base_url= openrouter_base_url)

gemini_model = OpenAIChatCompletionsModel(openai_client= gemini_client, model= "gemini-2.5-flash-lite")
openrouter_model = OpenAIChatCompletionsModel(openai_client= openrouter_client, model= "cohere/north-mini-code:free")

# agent = Agent(name="JokeTeller", instructions="You are a joke teller", model=gemini_model)
agent = Agent(name="JokeTeller", instructions="You are a joke teller", model=openrouter_model)

In [ ]:
result = await Runner.run(agent, input="Tell a joke about programmers")
print(result.final_output)

In [ ]:
result.to_input_list()

In [5]:
with trace ("Telling a joke"):
    result = await Runner.run(agent, input="Tell a joke about programmers")
print(result.final_output)

Why did the programmer quit his job?

Because he didn't get arrays!


### Now go and look at the trace

https://platform.openai.com/traces

In [ ]:
result = Runner.run_streamed(agent, input="Tell a joke about programmers")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

In [ ]:
@function_tool
def record_email_tool(email: str) -> str:
    """Use this tool to record that a user provided their email address"""

    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "/n")
    return "Email address is saved"

In [ ]:
record_email_tool

In [ ]:
record_email_tool.description

In [ ]:
record_email_tool.params_json_schema

In [ ]:
recorder = Agent(name= "EmailRecorder", instructions="You record user email upon request", model= openrouter_model, tools= [record_email_tool])

In [ ]:
result = await Runner.run(recorder, input="my email address is mahya.testy@test.com")
print(result.final_output)

## Part 3: Adding Memory

So that different Agent calls know about each other

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [ ]:
agent = Agent(name="Assistant", model=openrouter_model)

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Mahya")
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

## Memory approach 1 - just manually pass in the list of dicts

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Mahya")
print(response.final_output)

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

In [ ]:
response = await Runner.run(agent, next_input)
print(response.final_output)

## Another approach - use OpenAI Agents SDK built in SQLLite session

In [ ]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Mahya", session=session)
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>

## Digital Twin using OpenAI Agents SDK

In [54]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, Agent, Runner, function_tool, SQLiteSession
from pypdf import PdfReader
import gradio as gr
load_dotenv(override=True) 

True

In [19]:
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter_base_url = "https://openrouter.ai/api/v1"

openrouter_client = AsyncOpenAI(api_key= openrouter_api_key, base_url= openrouter_base_url)
openrouter_model = OpenAIChatCompletionsModel(openai_client= openrouter_client, model="cohere/north-mini-code:free")

In [ ]:
resume = ""
summary = ""
resumePdf = PdfReader("../1_foundations/twin/Resume.pdf")
for page in resumePdf.pages:
    text = page.extract_text()
    if text:
        resume += text

with open("../1_foundations/twin/summary.txt", "r", encoding= "utf-8") as f:
    summary = f.read()

twin_system_prompt = f"""
    You are my digital twin. When some people want to talk to me, you appear as my digital twin and answer the questions they ask about me.
    Keep in mind that these people should understand that you are my digital twin who answers their questions and not me.

    My resume is here: /n/n {resume} /n which includes my education, the universities I studied at,
    the research I did, my skills and my interests in the field of work and research.

    Also, a short summary of my characteristics when talking to people is here: /n/n {summary} 
    In fact, my model of answering people and some personality traits that you, as my digital twin, should consider when answering people.
    
    People who want to talk to me will first talk to you, who are my digital twin, and based on some general information that I have provided you, 
    you can answer their questions. 
    Speak to people in the same tone they speak to you. If they speak very politely and formally,respond in a formal and polite manner. 
    If they speak in a friendly and conversational tone, speak to them in the same friendly and friendly tone. 
    If they use swear words, use the same and respond to them in the same tone and expression. 
    If they speak to you in English, respond in English and if they speak to you in Persian, respond in Persian. 
    Let people ask you their own questions and it is better not to keep asking them what you can do for them.
    If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.
    """.strip()

In [32]:
@function_tool
def record_unknown_question(question):
    """Use this tool to record a question that you don't know the answer"""
    with open("unknown_question.txt", "a", unicode="utf-8") as f:
        f.write(question + "/n")
    return "Unknown question is saved to the file"

@function_tool
def record_email(email, name):
    """Use this tool to record an email of the user if it provided"""
    with open("Email.txt", "a", unicode="utf-8") as f:
        f.write(f"Name: {name} , Email: {email} /n")
    return "Name and email is saved"


In [33]:
digital_twin_agent = Agent(name= "MahyaDGTwin", instructions= twin_system_prompt, model= openrouter_model, tools=[record_unknown_question, record_email])

In [59]:
session = SQLiteSession("12346")
async def chat(message):
    response = await Runner.run(starting_agent= digital_twin_agent, input= message, session=session)
    return response.final_output

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)